# Smart Keyboard: train and evaluate on Google Colab

Runs the whole pipeline of the repository on a free GPU. Before you start:
**Runtime → Change runtime type → T4 GPU**.

Every step saves its output to Google Drive, so if Colab disconnects you can re-run the setup cells and
continue from the step that was interrupted.

## 1. Setup

In [2]:
from google.colab import drive


REPO_URL = 'https://github.com/Enes-Ozbek/smart-keyboard.git'   # <- change if your repo has another name
SAVE = '/content/save'           # artifacts and results live here

In [3]:
import os
if not os.path.exists('/content/smart-keyboard'):
    !git clone $REPO_URL /content/smart-keyboard
%cd /content/smart-keyboard
!git pull
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into '/content/smart-keyboard'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 26 (delta 0), reused 26 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 33.58 KiB | 8.40 MiB/s, done.
/content/smart-keyboard
Already up to date.
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [4]:
# every command below uses the same folders on Google Drive
ARGS = f'--artifacts-dir {SAVE}/artifacts --results-dir {SAVE}/results'

## 2. Data and the classical baseline (area B)

In [5]:
!python -m keyboard.cli prepare $ARGS


######## prepare ########
README.md: 100% 10.5k/10.5k [00:00<00:00, 29.9MB/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:   0% 0.00/733k [00:00<?, ?B/s]
wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:  66% 483k/733k [00:00<00:00, 850kB/s, 27.2kB/s  ]
wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes: 100% 732k/732k [00:00<00:00, 1.00MB/s, 71.2kB/s  ]
wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file: 100% 733k/733k [00:00<00:00, 1.00MB/s, 71.5kB/s  ]

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:  95% 6.01M/6.36M [00:00<00:00, 11.5MB/s]
wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes: 100% 6.35M/6.35M [00:00<00:00, 10.6MB/s,  623kB/s  ]
wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file: 100% 6.36M/6.36M [00:00<00:00, 10.6MB/s,  624kB/s  ]

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:   0% 0.00/657k [00:00<?, ?B/s]
wikitext-2-raw-v1/validation-00000-of-00(

In [6]:
!python -m keyboard.cli bpe $ARGS


######## bpe ########
[00:00:00] Tokenize words                 ██████████████████ 66145    /    66145
[00:00:00] Count pairs                    ██████████████████ 66145    /    66145
[00:00:02] Compute merges                 ██████████████████ 6360     /     6360
BPE: 8000 pieces -> /content/save/artifacts/bpe/tokenizer.json
######## bpe done in 5s


In [7]:
!python -m keyboard.cli ngram $ARGS


######## ngram ########
n-gram: 2,591,189 training tokens, distinct n-grams {1: 7502, 2: 530903, 3: 1565287, 4: 2183051} -> /content/save/artifacts/ngram/ngram4.npz
######## ngram done in 7s


## 3. Feedforward neural language model (area A)

In [8]:
!python -m keyboard.cli ffnn $ARGS


######## ffnn ########
FFNN: 5,456,192 parameters, 2,591,188 training windows, device=cuda
    epoch 1: 256,000/2,591,188 windows, train ppl 1183.7 (3s)
    epoch 1: 512,000/2,591,188 windows, train ppl 946.6 (5s)
    epoch 1: 768,000/2,591,188 windows, train ppl 838.3 (8s)
    epoch 1: 1,024,000/2,591,188 windows, train ppl 764.1 (10s)
    epoch 1: 1,280,000/2,591,188 windows, train ppl 710.0 (12s)
    epoch 1: 1,536,000/2,591,188 windows, train ppl 667.1 (15s)
    epoch 1: 1,792,000/2,591,188 windows, train ppl 631.6 (17s)
    epoch 1: 2,048,000/2,591,188 windows, train ppl 602.6 (20s)
    epoch 1: 2,304,000/2,591,188 windows, train ppl 577.7 (22s)
    epoch 1: 2,560,000/2,591,188 windows, train ppl 556.3 (25s)
  epoch 1: train ppl 553.7  valid ppl 384.0  lr 0.1  (26s)
    epoch 2: 256,000/2,591,188 windows, train ppl 331.9 (2s)
    epoch 2: 512,000/2,591,188 windows, train ppl 331.8 (5s)
    epoch 2: 768,000/2,591,188 windows, train ppl 330.7 (8s)
    epoch 2: 1,024,000/2,591,188 w

## 4. Fine-tune GPT-2 small (area C)

In [9]:
!python -m keyboard.cli gpt2 $ARGS


######## gpt2 ########
config.json: 100% 665/665 [00:00<00:00, 2.58MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 115kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 28.9MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 47.5MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 34.3MB/s]

model.safetensors: downloading bytes:  30% 166M/548M [00:01<00:02, 188MB/s, 14.1MB/s  ]
model.safetensors: downloading bytes:  78% 428M/548M [00:02<00:00, 273MB/s, 36.9MB/s  ]
model.safetensors: downloading bytes:  84% 463M/548M [00:02<00:00, 195MB/s, 40.5MB/s  ]
model.safetensors: reconstructing file:  86% 469M/548M [00:06<00:01, 62.7MB/s, 24.8MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:08<00:00, 58.2MB/s, 40.5MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:08<00:00, 67.3MB/s, 37.0MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 5997.25it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 684kB/s]
GPT-2: base gpt2, 2,345,472 training tokens, device

## 5. INT8 quantization (area D)

In [10]:
!python -m keyboard.cli quantize $ARGS


######## quantize ########
Loading weights: 100% 148/148 [00:00<00:00, 4122.52it/s]
INT8: 498 MB -> 282 MB  (/content/save/artifacts/gpt2-int8)
######## quantize done in 11s


## 6. Evaluate all models on the test split

Perplexity of the FP32 models uses the GPU; top-3 accuracy and latency run on the CPU of this Colab machine
(written under the table). This step takes a while because every model makes 1,000 timed predictions.
If the runtime was reset, run `prepare` again first (it recreates `data/`).

In [11]:
!python -m keyboard.cli evaluate $ARGS
print(open(f'{SAVE}/results/results.md').read())


######## evaluate ########

== ngram-addone: 4-gram add-one (counts)
   perplexity per word 87292.3 (per token 6382.1, 1s)
   top-3 accuracy 0.128, latency 3.9 ms (mean of 1000 runs)

== ngram-kn: 4-gram Kneser-Ney (counts)
   perplexity per word 660.5 (per token 148.4, 1s)
   top-3 accuracy 0.304, latency 5.2 ms (mean of 1000 runs)

== ffnn: Feedforward NN (FP32)
   perplexity per word 1244.9 (per token 241.8, 1s)
   top-3 accuracy 0.265, latency 9.5 ms (mean of 1000 runs)
Loading weights: 100% 148/148 [00:00<00:00, 3990.62it/s]

== gpt2-fp32: GPT-2 fine-tuned (FP32)
   perplexity per word 40.2 (per token 24.4, 61s)
   top-3 accuracy 0.477, latency 356.3 ms (mean of 1000 runs)
/usr/local/lib/python3.13/dist-packages/torch/_utils.py:455: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_stor

## 7. Upload the weights to the Hugging Face Hub (R7: no weights in git)

Create a write token at https://huggingface.co/settings/tokens first.

In [12]:
from huggingface_hub import notebook_login
notebook_login()

In [13]:
HF_REPO = 'USER/smart-keyboard-models'    # <- your Hugging Face user name
!python -m keyboard.cli upload $ARGS --models-repo $HF_REPO

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_http.py", line 836, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/smart-keyboard/keyboard/cli.py", line 79, in <module>
    main()
    ~~~~^^
  File "/content/smart-keyboard/keyboard/cli.py", line 70, in main
    upload(cfg)
    ~~~~~~^^^^^
  File "/content/smart-key

## 8. Try the app (public link)

In [ ]:
!python app.py --share --artifacts-dir $SAVE/artifacts

/usr/local/lib/python3.13/dist-packages/torch/_utils.py:455: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,
loaded gpt2-int8
Loading weights: 100% 148/148 [00:00<00:00, 5909.44it/s]
loaded gpt2-fp32
loaded ffnn
loaded ngram-kn
loaded ngram-addone
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b872bb962b224842cd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
